# Topic Modeling with BERTopic — Parameter Search & Evaluation

Loads pre-computed embeddings for the **iGEM teams** and **SynBio papers** datasets,
runs a grid search over key UMAP/HDBSCAN parameters, evaluates each configuration
using **topic coherence** ($C_v$), **topic diversity**, and **DBCV**, picks the best
model per dataset, and saves the results.

In [24]:
import os
from itertools import product

import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

In [25]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── PARAMETER GRID ────────────────────────────────────────────────────────────
PARAM_GRID_TEAMS = {
    'min_cluster_size': [8, 10, 15, 20],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

PARAM_GRID_PAPERS = {
    'min_cluster_size': [20, 30, 40, 50],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

## 1. Load embeddings and corpora

In [26]:
teams_embeddings  = np.load(os.path.join(EMBEDDINGS_DIR, 'teams_embeddings.npy'))
papers_embeddings = np.load(os.path.join(EMBEDDINGS_DIR, 'papers_embeddings.npy'))

teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'), sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

teams_docs  = teams_corpus['text'].tolist()
papers_docs = papers_corpus['text'].tolist()

print(f'Teams:  {teams_embeddings.shape[0]:,} docs, {teams_embeddings.shape[1]} dims')
print(f'Papers: {papers_embeddings.shape[0]:,} docs, {papers_embeddings.shape[1]} dims')

assert len(teams_docs) == teams_embeddings.shape[0]
assert len(papers_docs) == papers_embeddings.shape[0]

Teams:  4,548 docs, 384 dims
Papers: 33,625 docs, 384 dims


## 2. Helpers: model fitting and evaluation metrics

In [27]:
def fit_topic_model(
    docs: list[str],
    embeddings: np.ndarray,
    min_cluster_size: int = 15,
    umap_n_neighbors: int = 15,
    umap_n_components: int = 5,
) -> tuple[BERTopic, list[int], np.ndarray]:
    """Fit a BERTopic model using UMAP + HDBSCAN."""
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=SEED,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
        gen_min_span_tree=True,
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=min_cluster_size,
        n_gram_range=(1, 3),
        language='english',
        calculate_probabilities=True,
        verbose=False,
    )
    topics, probs = topic_model.fit_transform(docs, embeddings)
    return topic_model, topics, probs

In [28]:
def coherence_cv(topic_model: BERTopic, docs: list[str], top_n: int = 10) -> float:
    """Compute C_v coherence over the top-N words of each non-outlier topic.

    BERTopic may produce multi-word n-grams (e.g. 'gene expression').
    We split those into individual tokens so every term is present in the
    unigram dictionary that Gensim builds from the corpus.
    """
    topics = topic_model.get_topics()
    topic_words = []
    for t in sorted(topics):
        if t == -1:
            continue
        words = []
        for word, _ in topics[t][:top_n]:
            words.extend(word.lower().split())  # split n-grams into unigrams
        topic_words.append(words)
    if not topic_words:
        return float('nan')
    tokenized = [doc.lower().split() for doc in docs]
    dictionary = Dictionary(tokenized)
    # Keep only tokens that exist in the dictionary
    topic_words = [
        [w for w in tw if w in dictionary.token2id]
        for tw in topic_words
    ]
    topic_words = [tw for tw in topic_words if len(tw) >= 2]
    if not topic_words:
        return float('nan')
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v',
    )
    return cm.get_coherence()


def topic_diversity(topic_model: BERTopic, top_n: int = 10) -> float:
    """Fraction of unique words across all topics' top-N word lists."""
    topics = topic_model.get_topics()
    all_words = [
        word
        for t in sorted(topics) if t != -1
        for word, _ in topics[t][:top_n]
    ]
    return len(set(all_words)) / len(all_words) if all_words else float('nan')


def dbcv_score(topic_model: BERTopic) -> float:
    """HDBSCAN relative validity (DBCV). Higher is better, range [-1, 1]."""
    return topic_model.hdbscan_model.relative_validity_

In [29]:
def grid_search(
    docs: list[str],
    embeddings: np.ndarray,
    param_grid: dict,
    label: str = '',
) -> pd.DataFrame:
    """Run all parameter combinations, evaluate each, return results DataFrame."""
    combos = list(product(
        param_grid['min_cluster_size'],
        param_grid['umap_n_neighbors'],
        param_grid['umap_n_components'],
    ))
    n_total = len(combos)
    rows = []
    best_score = -np.inf
    best_model_data = None

    for i, (mcs, nn, nc) in enumerate(combos, 1):
        print(f'[{label}] {i}/{n_total}  mcs={mcs}, nn={nn}, nc={nc} ... ', end='', flush=True)
        model, topics, probs = fit_topic_model(
            docs, embeddings,
            min_cluster_size=mcs, umap_n_neighbors=nn, umap_n_components=nc,
        )
        n_topics = model.get_topic_info().Topic.max() + 1
        outlier_frac = (np.array(topics) == -1).mean()
        cv  = coherence_cv(model, docs)
        div = topic_diversity(model)
        dbcv = dbcv_score(model)

        row = {
            'min_cluster_size': mcs,
            'n_neighbors': nn,
            'n_components': nc,
            'n_topics': n_topics,
            'outlier_frac': round(outlier_frac, 4),
            'coherence_cv': round(cv, 4),
            'diversity': round(div, 4),
            'dbcv': round(dbcv, 4),
        }
        rows.append(row)
        print(f'topics={n_topics}, C_v={cv:.4f}, div={div:.4f}, DBCV={dbcv:.4f}')

        # Track best model by C_v (primary), break ties by diversity
        if cv > best_score or (cv == best_score and div > (best_model_data or {}).get('diversity', -1)):
            best_score = cv
            best_model_data = {
                'model': model, 'topics': topics, 'probs': probs,
                'diversity': div, **row,
            }

    df = pd.DataFrame(rows).sort_values('coherence_cv', ascending=False).reset_index(drop=True)
    return df, best_model_data

## 3. Grid search: iGEM Teams

In [30]:
teams_results, teams_best = grid_search(
    teams_docs, teams_embeddings, PARAM_GRID_TEAMS, label='Teams'
)
teams_results

[Teams] 1/24  mcs=8, nn=10, nc=5 ... topics=127, C_v=0.5803, div=0.7024, DBCV=0.2897
[Teams] 2/24  mcs=8, nn=10, nc=10 ... topics=121, C_v=0.5704, div=0.6777, DBCV=0.2260
[Teams] 3/24  mcs=8, nn=15, nc=5 ... topics=107, C_v=0.5723, div=0.6364, DBCV=0.2422
[Teams] 4/24  mcs=8, nn=15, nc=10 ... topics=99, C_v=0.5621, div=0.6394, DBCV=0.2513
[Teams] 5/24  mcs=8, nn=25, nc=5 ... topics=103, C_v=0.5841, div=0.6437, DBCV=0.1633
[Teams] 6/24  mcs=8, nn=25, nc=10 ... topics=91, C_v=0.5751, div=0.6220, DBCV=0.1881
[Teams] 7/24  mcs=10, nn=10, nc=5 ... topics=100, C_v=0.5554, div=0.6000, DBCV=0.2356
[Teams] 8/24  mcs=10, nn=10, nc=10 ... topics=96, C_v=0.5715, div=0.6104, DBCV=0.2389
[Teams] 9/24  mcs=10, nn=15, nc=5 ... topics=87, C_v=0.5491, div=0.5816, DBCV=0.2566
[Teams] 10/24  mcs=10, nn=15, nc=10 ... topics=75, C_v=0.5394, div=0.5653, DBCV=0.2394
[Teams] 11/24  mcs=10, nn=25, nc=5 ... topics=87, C_v=0.5742, div=0.5943, DBCV=0.1898
[Teams] 12/24  mcs=10, nn=25, nc=10 ... topics=79, C_v=0.56

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,8,25,5,103,0.4017,0.5841,0.6437,0.1633
1,8,10,5,127,0.3199,0.5803,0.7024,0.2897
2,8,25,10,91,0.3815,0.5751,0.6220,0.1881
3,10,25,5,87,0.4077,0.5742,0.5943,0.1898
4,8,15,5,107,0.3382,0.5723,0.6364,0.2422
5,10,10,10,96,0.3043,0.5715,0.6104,0.2389
6,8,10,10,121,0.2861,0.5704,0.6777,0.2260
7,10,25,10,79,0.3986,0.5630,0.5734,0.2243
8,8,15,10,99,0.3278,0.5621,0.6394,0.2513
9,10,10,5,100,0.3349,0.5554,0.6000,0.2356


In [31]:
print('Best Teams configuration:')
print(f"  min_cluster_size = {teams_best['min_cluster_size']}")
print(f"  n_neighbors      = {teams_best['n_neighbors']}")
print(f"  n_components     = {teams_best['n_components']}")
print(f"  n_topics         = {teams_best['n_topics']}")
print(f"  C_v              = {teams_best['coherence_cv']}")
print(f"  diversity        = {teams_best['diversity']}")
print(f"  DBCV             = {teams_best['dbcv']}")
print(f"  outlier_frac     = {teams_best['outlier_frac']}")

Best Teams configuration:
  min_cluster_size = 8
  n_neighbors      = 25
  n_components     = 5
  n_topics         = 103
  C_v              = 0.5841
  diversity        = 0.6437
  DBCV             = 0.1633
  outlier_frac     = 0.4017


## 4. Grid search: SynBio Papers

In [32]:
papers_results, papers_best = grid_search(
    papers_docs, papers_embeddings, PARAM_GRID_PAPERS, label='Papers'
)
papers_results

[Papers] 1/24  mcs=20, nn=10, nc=5 ... topics=247, C_v=0.6743, div=0.7186, DBCV=0.2197
[Papers] 2/24  mcs=20, nn=10, nc=10 ... topics=247, C_v=0.6760, div=0.7308, DBCV=0.2113
[Papers] 3/24  mcs=20, nn=15, nc=5 ... topics=250, C_v=0.6792, div=0.7308, DBCV=0.1921
[Papers] 4/24  mcs=20, nn=15, nc=10 ... topics=234, C_v=0.6839, div=0.7299, DBCV=0.2055
[Papers] 5/24  mcs=20, nn=25, nc=5 ... topics=202, C_v=0.6762, div=0.7332, DBCV=0.1816
[Papers] 6/24  mcs=20, nn=25, nc=10 ... topics=208, C_v=0.6782, div=0.7317, DBCV=0.2568
[Papers] 7/24  mcs=30, nn=10, nc=5 ... topics=169, C_v=0.6517, div=0.6639, DBCV=0.2161
[Papers] 8/24  mcs=30, nn=10, nc=10 ... topics=160, C_v=0.6476, div=0.6675, DBCV=0.2194
[Papers] 9/24  mcs=30, nn=15, nc=5 ... topics=164, C_v=0.6570, div=0.6726, DBCV=0.2025
[Papers] 10/24  mcs=30, nn=15, nc=10 ... topics=164, C_v=0.6664, div=0.6902, DBCV=0.2106
[Papers] 11/24  mcs=30, nn=25, nc=5 ... topics=141, C_v=0.6458, div=0.6709, DBCV=0.2167
[Papers] 12/24  mcs=30, nn=25, nc=10

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,20,15,10,234,0.4745,0.6839,0.7299,0.2055
1,20,15,5,250,0.4894,0.6792,0.7308,0.1921
2,20,25,10,208,0.5140,0.6782,0.7317,0.2568
3,20,25,5,202,0.5054,0.6762,0.7332,0.1816
4,20,10,10,247,0.4182,0.6760,0.7308,0.2113
5,20,10,5,247,0.4470,0.6743,0.7186,0.2197
6,30,15,10,164,0.4986,0.6664,0.6902,0.2106
7,30,15,5,164,0.5072,0.6570,0.6726,0.2025
8,30,10,5,169,0.4675,0.6517,0.6639,0.2161
9,30,10,10,160,0.4368,0.6476,0.6675,0.2194


In [33]:
print('Best Papers configuration:')
print(f"  min_cluster_size = {papers_best['min_cluster_size']}")
print(f"  n_neighbors      = {papers_best['n_neighbors']}")
print(f"  n_components     = {papers_best['n_components']}")
print(f"  n_topics         = {papers_best['n_topics']}")
print(f"  C_v              = {papers_best['coherence_cv']}")
print(f"  diversity        = {papers_best['diversity']}")
print(f"  DBCV             = {papers_best['dbcv']}")
print(f"  outlier_frac     = {papers_best['outlier_frac']}")

Best Papers configuration:
  min_cluster_size = 20
  n_neighbors      = 15
  n_components     = 10
  n_topics         = 234
  C_v              = 0.6839
  diversity        = 0.7299
  DBCV             = 0.2055
  outlier_frac     = 0.4745


## 5. Save best models and outputs

In [34]:
teams_model  = teams_best['model']
papers_model = papers_best['model']
teams_topics  = teams_best['topics']
papers_topics = papers_best['topics']

# ── Save models ───────────────────────────────────────────────────────────────
teams_model.save(os.path.join(MODELS_DIR, 'teams_topic_model'))
papers_model.save(os.path.join(MODELS_DIR, 'papers_topic_model'))

# ── Save topic info summaries ─────────────────────────────────────────────────
teams_info  = teams_model.get_topic_info()
papers_info = papers_model.get_topic_info()

teams_info.to_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'), sep='\t', index=False)
papers_info.to_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t', index=False)

# ── Save document-level topic assignments ─────────────────────────────────────
teams_corpus['topic'] = teams_topics
teams_corpus[['UT', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'teams_doc_topics.txt'), sep='\t', index=False
)
papers_corpus['topic'] = papers_topics
papers_corpus[['id', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t', index=False
)

# ── Save grid search results ─────────────────────────────────────────────────
teams_results.to_csv(os.path.join(MODELS_DIR, 'teams_grid_search.txt'), sep='\t', index=False)
papers_results.to_csv(os.path.join(MODELS_DIR, 'papers_grid_search.txt'), sep='\t', index=False)

print(f'All outputs saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

2026-03-26 14:27:11,905 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-03-26 14:27:16,951 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


All outputs saved to ../assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
